In [4]:
import pandas as pd
import numpy as np
import tensorflow as tf
import os
import keras
from keras import layers
from tensorflow.keras import backend as K
import matplotlib.pyplot as plt
from IPython import display
from jiwer import wer
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')

data_url = "https://data.keithito.com/data/speech/LJSpeech-1.1.tar.bz2"
data_path = keras.utils.get_file("LJSpeech-1.1", data_url, untar=True)
metadata_path = os.path.join(data_path, "metadata.csv")

if not os.path.exists(metadata_path):
    metadata_path = os.path.join(data_path, "LJSpeech-1.1", "metadata.csv")
    wavs_path = os.path.join(data_path, "LJSpeech-1.1", "wavs") + os.sep
else:
    wavs_path = os.path.join(data_path, "wavs") + os.sep

print(f"Шлях до метаданих: {metadata_path}")

metadata_df = pd.read_csv(metadata_path, sep="|", header=None, quoting=3)
metadata_df.columns = ["file_name", "transcription", "normalized_transcription"]
metadata_df = metadata_df[["file_name", "normalized_transcription"]]
metadata_df = metadata_df.dropna()
metadata_df = metadata_df.sample(frac=1, random_state=42).reset_index(drop=True)

metadata_df = metadata_df[:4000]

split = int(len(metadata_df) * 0.90)
df_train = metadata_df[:split]
df_val = metadata_df[split:]

print(f"Size of the training set: {len(df_train)}")
print(f"Size of the validation set: {len(df_val)}")

characters = [x for x in "abcdefghijklmnopqrstuvwxyz'?! "]
char_to_num = keras.layers.StringLookup(vocabulary=characters, oov_token="")
num_to_char = keras.layers.StringLookup(vocabulary=char_to_num.get_vocabulary(), oov_token="", invert=True)

frame_length = 256
frame_step = 160
fft_length = 384


def encode_single_sample(wav_file, label):
    file = tf.io.read_file(wavs_path + wav_file + ".wav")
    audio, _ = tf.audio.decode_wav(file)
    audio = tf.squeeze(audio, axis=-1)
    audio = tf.cast(audio, tf.float32)

    stft_output = tf.signal.stft(
        audio, frame_length=frame_length, frame_step=frame_step, fft_length=fft_length
    )
    spectrogram = tf.abs(stft_output)
    spectrogram = tf.math.pow(spectrogram, 0.5)

    means = tf.math.reduce_mean(spectrogram)
    stddevs = tf.math.reduce_std(spectrogram)
    spectrogram = (spectrogram - means) / (stddevs + 1e-10)

    label = tf.strings.lower(label)
    label = tf.strings.unicode_split(label, input_encoding="UTF-8")
    label = char_to_num(label)
    return spectrogram, label


batch_size = 32
train_dataset = tf.data.Dataset.from_tensor_slices(
    (list(df_train["file_name"]), list(df_train["normalized_transcription"])))
train_dataset = (
    train_dataset.map(encode_single_sample, num_parallel_calls=tf.data.AUTOTUNE)
    .cache()
    .padded_batch(batch_size, padded_shapes=([None, fft_length // 2 + 1], [None]))
    .prefetch(buffer_size=tf.data.AUTOTUNE)
)

validation_dataset = tf.data.Dataset.from_tensor_slices(
    (list(df_val["file_name"]), list(df_val["normalized_transcription"])))
validation_dataset = (
    validation_dataset.map(encode_single_sample, num_parallel_calls=tf.data.AUTOTUNE)
    .cache()
    .padded_batch(batch_size, padded_shapes=([None, fft_length // 2 + 1], [None]))
    .prefetch(buffer_size=tf.data.AUTOTUNE)
)


def CTCLoss(y_true, y_pred):
    batch_len = tf.shape(y_true)[0]
    input_length = tf.shape(y_pred)[1]

    label_length = tf.math.count_nonzero(y_true, axis=-1, dtype=tf.int32)
    input_length = input_length * tf.ones(shape=(batch_len,), dtype=tf.int32)

    loss = tf.nn.ctc_loss(
        labels=tf.cast(y_true, tf.int32),
        logits=y_pred,
        label_length=label_length,
        logit_length=input_length,
        logits_time_major=False,
        blank_index=0
    )
    return tf.reduce_mean(loss)


def build_model(input_dim, output_dim, rnn_layers=2, rnn_units=256):
    input_spectrogram = layers.Input((None, input_dim), name="input")
    x = layers.Reshape((-1, input_dim, 1), name="expand_dim")(input_spectrogram)

    x = layers.Conv2D(32, kernel_size=[11, 21], strides=[2, 2], padding="same", use_bias=False, name="conv_1")(x)
    x = layers.BatchNormalization(name="conv_1_bn")(x)
    x = layers.ReLU(name="conv_1_relu")(x)

    x = layers.Reshape((-1, x.shape[-2] * x.shape[-1]))(x)

    for i in range(1, rnn_layers + 1):
        recurrent = layers.GRU(
            units=rnn_units, activation="tanh", recurrent_activation="sigmoid",
            use_bias=True, return_sequences=True, reset_after=True, name=f"gru_{i}"
        )
        x = layers.Bidirectional(recurrent, name=f"bidirectional_{i}", merge_mode="concat")(x)
        if i < rnn_layers:
            x = layers.Dropout(rate=0.5)(x)

    x = layers.Dense(units=rnn_units * 2, name="dense_1")(x)
    x = layers.ReLU(name="dense_1_relu")(x)
    x = layers.Dropout(rate=0.5)(x)

    output = layers.Dense(units=output_dim + 1, activation="linear")(x)

    model = keras.Model(input_spectrogram, output, name="DeepSpeech_Balanced")

    opt = keras.optimizers.Adam(
        learning_rate=5e-5,
        clipnorm=1.0
    )
    model.compile(optimizer=opt, loss=CTCLoss)
    return model

K.clear_session()
model = build_model(input_dim=fft_length // 2 + 1, output_dim=char_to_num.vocabulary_size())


def decode_batch_predictions(pred):
    pred = tf.cast(pred, tf.float32)

    pred = tf.nn.softmax(pred)

    input_len = np.ones(pred.shape[0]) * pred.shape[1]
    decoded = K.ctc_decode(y_pred=pred, input_length=tf.cast(input_len, tf.int32), greedy=True)
    decoded_sequences = decoded[0][0].numpy()

    output_text = []
    for sequence in decoded_sequences:
        sequence = sequence[sequence > 0]
        text = tf.strings.reduce_join(num_to_char(sequence)).numpy().decode("utf-8")
        output_text.append(text)
    return output_text

class CallbackEval(keras.callbacks.Callback):
    def __init__(self, dataset):
        super().__init__()
        self.dataset = dataset

    def on_epoch_end(self, epoch: int, logs=None):
        predictions = []
        targets = []
        for i, batch in enumerate(self.dataset):
            if i >= 3:
                break
            X, y = batch
            batch_predictions = model.predict(X, verbose=0)
            batch_predictions = decode_batch_predictions(batch_predictions)
            predictions.extend(batch_predictions)

            for label in y:
                label = label[label > 0]
                label = tf.strings.reduce_join(num_to_char(label)).numpy().decode("utf-8")
                targets.append(label)

        wer_score = wer(targets, predictions)
        print("-" * 100)
        print(f"Word Error Rate: {wer_score:.4f}")
        print("-" * 100)
        for i in np.random.randint(0, len(predictions), 2):
            print(f"Target    : {targets[i]}")
            print(f"Prediction: {predictions[i]}")
            print("-" * 100)


epochs = 50
validation_callback = CallbackEval(validation_dataset)

lr_schedule = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)
checkpoint_callback = keras.callbacks.ModelCheckpoint(
    filepath="best_deepspeech_model.keras",
    monitor="val_loss",
    save_best_only=True,
    save_weights_only=False,
    verbose=1
)

history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=epochs,
    callbacks=[validation_callback, lr_schedule, checkpoint_callback]
)

model.save("final_deepspeech_model.keras")
print("Фінальну модель успішно збережено у 'final_deepspeech_model.keras'")

predictions = []
targets = []
for batch in validation_dataset:
    X, y = batch
    batch_predictions = model.predict(X)
    batch_predictions = decode_batch_predictions(batch_predictions)
    predictions.extend(batch_predictions)
    for label in y:
        label = tf.strings.reduce_join(num_to_char(label)).numpy().decode("utf-8")
        targets.append(label)
wer_score = wer(targets, predictions)
print("-" * 100)
print(f"Word Error Rate: {wer_score:.4f}")
print("-" * 100)
for i in np.random.randint(0, len(predictions), 5):
    print(f"Target    : {targets[i]}")
    print(f"Prediction: {predictions[i]}")
    print("-" * 100)

Шлях до метаданих: /root/.keras/datasets/LJSpeech-1_extracted/LJSpeech-1.1/metadata.csv
Size of the training set: 3600
Size of the validation set: 400
Epoch 1/50
113/113 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - loss: 723.5337----------------------------------------------------------------------------------------------------
Word Error Rate: 1.0000
----------------------------------------------------------------------------------------------------
Target    : economic political or military crisis internal or external
Prediction: 
----------------------------------------------------------------------------------------------------
Target    : the objects it had in view were set forth in one of its earliest meetings
Prediction: 
----------------------------------------------------------------------------------------------------

Epoch 1: val_loss improved from None to 425.44159, saving model to best_deepspeech_model.keras

Epoch 1: finished saving model to best_deepspeech_model.keras
113/113 ━━━━

KeyboardInterrupt: 

In [5]:
import tensorflow as tf
from tensorflow import keras


K.clear_session()

def CTCLoss(y_true, y_pred):
    batch_len = tf.shape(y_true)[0]
    input_length = tf.shape(y_pred)[1]

    label_length = tf.math.count_nonzero(y_true, axis=-1, dtype=tf.int32)
    input_length = input_length * tf.ones(shape=(batch_len,), dtype=tf.int32)

    loss = tf.nn.ctc_loss(
        labels=tf.cast(y_true, tf.int32),
        logits=y_pred,
        label_length=label_length,
        logit_length=input_length,
        logits_time_major=False,
        blank_index=0
    )
    return tf.reduce_mean(loss)


model = keras.models.load_model(
    "best_deepspeech_model.keras",
    custom_objects={"CTCLoss": CTCLoss}
)
print("Найкращу модель успішно завантажено!")

opt = keras.optimizers.Adam(learning_rate=1e-5, clipnorm=0.5)
model.compile(optimizer=opt, loss=CTCLoss)

lr_schedule = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-7,
    verbose=1
)

checkpoint_callback = keras.callbacks.ModelCheckpoint(
    filepath="best_deepspeech_model.keras",
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=6,
    restore_best_weights=True,
    verbose=1
)

print("\nПродовжуємо навчання з обережнішими налаштуваннями...")
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=20,
    callbacks=[validation_callback, lr_schedule, checkpoint_callback, early_stopping]
)

Найкращу модель успішно завантажено!

Продовжуємо навчання з обережнішими налаштуваннями...
Epoch 1/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - loss: 226.1773----------------------------------------------------------------------------------------------------
Word Error Rate: 0.9782
----------------------------------------------------------------------------------------------------
Target    : had not quite disappeared i will mention two cases of this class one accompanied with piracy on the high seas
Prediction: no is nntopss o thi w ao  wprso is
----------------------------------------------------------------------------------------------------
Target    : and daniel lujan aged twentysix was number four
Prediction: nnl siots wsrfo
----------------------------------------------------------------------------------------------------

Epoch 1: val_loss improved from None to 206.49866, saving model to best_deepspeech_model.keras

Epoch 1: finished saving model to best_deepspeech_model.ker

KeyboardInterrupt: 